# Notebook 01: Data Cleaning and Data Dictionary

This notebook loads the raw UCI diabetes readmission dataset, reviews structure and missingness, creates the binary 30-day readmission target, documents key data quality issues, and exports a cleaned starter dataset for later EDA and feature engineering.

The project uses encounter-level modeling. Each row represents one hospital encounter.

## 1. Imports

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

## 2. Project Paths

In [2]:
# Allows the notebook to run whether opened from the project root or the notebooks folder.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
OUTPUTS = PROJECT_ROOT / "outputs"
MODEL_RESULTS = OUTPUTS / "model_results"

for path in [DATA_RAW, DATA_PROCESSED, OUTPUTS, MODEL_RESULTS]:
    path.mkdir(parents=True, exist_ok=True)

print("Project root detected successfully.")
print("Project folder:", PROJECT_ROOT.name)

Project root detected successfully.
Project folder: 03_risk_stratification_intervention_prioritization


## 3. Load Raw Files

In [3]:
diabetes_path = DATA_RAW / "diabetic_data.csv"
mapping_path = DATA_RAW / "IDS_mapping.csv"

assert diabetes_path.exists(), f"Missing file: {diabetes_path}"
assert mapping_path.exists(), f"Missing file: {mapping_path}"

df_raw = pd.read_csv(diabetes_path)
ids_mapping = pd.read_csv(mapping_path)

print("Raw diabetes shape:", df_raw.shape)
print("IDS mapping shape:", ids_mapping.shape)

Raw diabetes shape: (101766, 50)
IDS mapping shape: (67, 2)


## 4. Preview Data

In [4]:
display(df_raw.head())
display(ids_mapping.head(20))

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,?,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,?,?,1,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,?,?,59,0,18,0,0,0,276,250.01,255,9,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,?,?,11,5,13,2,0,1,648,250,V27,6,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,?,?,44,1,16,0,0,0,8,250.43,403,7,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,?,?,51,0,8,0,0,0,197,157,250,5,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO


,admission_type_id,description
0,1,Emergency
1,2,Urgent
2,3,Elective
3,4,Newborn
4,5,Not Available
5,6,NaN
6,7,Trauma Center
7,8,Not Mapped
8,NaN,NaN
9,discharge_disposition_id,description


## 5. Basic Structure

In [5]:
print("Rows:", df_raw.shape[0])
print("Columns:", df_raw.shape[1])

structure = pd.DataFrame({
    "column": df_raw.columns,
    "dtype": df_raw.dtypes.astype(str).values,
    "non_null_count": df_raw.notna().sum().values,
    "unique_count": df_raw.nunique(dropna=False).values
})

display(structure)

Rows: 101766
Columns: 50


,column,dtype,non_null_count,unique_count
0,encounter_id,int64,101766,101766
1,patient_nbr,int64,101766,71518
2,race,str,101766,6
3,gender,str,101766,3
4,age,str,101766,10
5,weight,str,101766,10
6,admission_type_id,int64,101766,8
7,discharge_disposition_id,int64,101766,26
8,admission_source_id,int64,101766,17
9,time_in_hospital,int64,101766,14


## 6. ID Quality Checks

In [6]:
id_checks = pd.DataFrame({
    "check": [
        "Total rows",
        "Unique encounter_id",
        "Duplicate encounter_id count",
        "Unique patient_nbr",
        "Repeated patient_nbr count"
    ],
    "value": [
        len(df_raw),
        df_raw["encounter_id"].nunique(),
        df_raw["encounter_id"].duplicated().sum(),
        df_raw["patient_nbr"].nunique(),
        len(df_raw) - df_raw["patient_nbr"].nunique()
    ]
})

display(id_checks)

,check,value
0,Total rows,101766
1,Unique encounter_id,101766
2,Duplicate encounter_id count,0
3,Unique patient_nbr,71518
4,Repeated patient_nbr count,30248


`encounter_id` identifies hospital encounters and should not be used as a predictive feature.

`patient_nbr` identifies patients and should not be used as a predictive feature.

This project keeps the encounter as the unit of analysis, meaning each row represents one hospital encounter. However, because the same patient can appear in multiple encounters, later modeling notebooks must not use a simple random row-level train/test split. A random split could place encounters from the same patient in both training and test sets, causing train/test contamination and inflated performance metrics.

Later modeling notebooks should use `patient_nbr` as the grouping variable during train/test splitting or cross-validation.

Recommended approaches:

- `GroupShuffleSplit` for a grouped train/test split
- `GroupKFold` or `StratifiedGroupKFold` for grouped cross-validation

The model should not use `encounter_id` or `patient_nbr` as predictive features.

## 7. Check ? missing codes

In [7]:
missing_code_counts = (df_raw == "?").sum().sort_values(ascending=False)

missing_code_summary = (
    missing_code_counts[missing_code_counts > 0]
    .rename("missing_code_count")
    .reset_index()
    .rename(columns={"index": "column"})
)

missing_code_summary["missing_code_pct"] = (
    missing_code_summary["missing_code_count"] / len(df_raw) * 100
).round(2)

display(missing_code_summary)

,column,missing_code_count,missing_code_pct
0,weight,98569,96.86
1,medical_specialty,49949,49.08
2,payer_code,40256,39.56
3,race,2273,2.23
4,diag_3,1423,1.40
5,diag_2,358,0.35
6,diag_1,21,0.02


## 8. Replace ? with NaN

In [8]:
# The UCI dataset uses "?" as a missing-value code, so convert it to real NaN values.
df_clean = df_raw.replace("?", np.nan).copy()

missing_summary = pd.DataFrame({
    "column": df_clean.columns,
    "dtype": df_clean.dtypes.astype(str).values,
    "missing_count": df_clean.isna().sum().values,
    "missing_pct": (df_clean.isna().mean().values),
    "unique_count": df_clean.nunique(dropna=False).values
}).sort_values("missing_pct", ascending=False)

display(missing_summary)

,column,dtype,missing_count,missing_pct,unique_count
5,weight,str,98569,0.968585,10
22,max_glu_serum,str,96420,0.947468,4
23,A1Cresult,str,84748,0.832773,4
11,medical_specialty,str,49949,0.490822,73
10,payer_code,str,40256,0.395574,18
2,race,str,2273,0.022336,6
20,diag_3,str,1423,0.013983,790
19,diag_2,str,358,0.003518,749
18,diag_1,str,21,0.000206,717
1,patient_nbr,int64,0,0.000000,71518


## 9. Identify high-missingness columns

In [9]:
# High-missingness columns are documented here, but dropping them is deferred to feature engineering.
high_missing_threshold = 0.40

high_missing_columns = missing_summary[
    missing_summary["missing_pct"] >= high_missing_threshold
].copy()

display(high_missing_columns)

,column,dtype,missing_count,missing_pct,unique_count
5,weight,str,98569,0.968585,10
22,max_glu_serum,str,96420,0.947468,4
23,A1Cresult,str,84748,0.832773,4
11,medical_specialty,str,49949,0.490822,73


Columns with high missingness are documented but not automatically removed in this notebook.

Dropping columns is a modeling decision and will be handled later during feature engineering.

## 10. Target Review

In [10]:
target_counts = (
    df_clean["readmitted"]
    .value_counts(dropna=False)
    .rename_axis("readmitted")
    .reset_index(name="count")
)

target_counts["pct"] = target_counts["count"] / len(df_clean)

display(target_counts)

,readmitted,count,pct
0,NO,54864,0.539119
1,>30,35545,0.349282
2,<30,11357,0.111599


## 11. Create Binary Target

In [11]:
expected_readmitted_values = {"<30", ">30", "NO"}
actual_readmitted_values = set(df_clean["readmitted"].dropna().unique())

unexpected_values = actual_readmitted_values - expected_readmitted_values
assert len(unexpected_values) == 0, f"Unexpected readmitted values: {unexpected_values}"

# Primary target: 1 means readmitted within 30 days; >30 and NO are treated as 0.
df_clean["readmitted_30d"] = np.where(df_clean["readmitted"] == "<30", 1, 0)

display(
    df_clean["readmitted_30d"]
    .value_counts()
    .rename_axis("readmitted_30d")
    .reset_index(name="count")
)

baseline_readmission_rate = df_clean["readmitted_30d"].mean()
print(f"Baseline 30-day readmission rate: {baseline_readmission_rate:.2%}")

,readmitted_30d,count
0,0,90409
1,1,11357


Baseline 30-day readmission rate: 11.16%


## 12. Sensitive Demographic Fields

In [12]:
# Sensitive demographic fields are retained for subgroup review, not automatically used for prediction.
sensitive_columns = ["race", "gender", "age"]

sensitive_summary = []

for col in sensitive_columns:
    if col in df_clean.columns:
        temp = (
            df_clean[col]
            .value_counts(dropna=False)
            .rename_axis(col)
            .reset_index(name="count")
        )
        temp["pct"] = temp["count"] / len(df_clean)
        temp["column"] = col
        sensitive_summary.append(temp)

sensitive_summary = pd.concat(sensitive_summary, ignore_index=True)

display(sensitive_summary)

,race,count,pct,column,gender,age
0,Caucasian,76099,0.747784,race,NaN,NaN
1,AfricanAmerican,19210,0.188766,race,NaN,NaN
2,NaN,2273,0.022336,race,NaN,NaN
3,Hispanic,2037,0.020017,race,NaN,NaN
4,Other,1506,0.014799,race,NaN,NaN
5,Asian,641,0.006299,race,NaN,NaN
6,NaN,54708,0.537586,gender,Female,NaN
7,NaN,47055,0.462384,gender,Male,NaN
8,NaN,3,0.000029,gender,Unknown/Invalid,NaN
9,NaN,26068,0.256156,age,NaN,[70-80)


Race, gender, and age are sensitive demographic fields.

They are retained for subgroup review and fairness monitoring, but their use as predictive features will be reviewed carefully in later notebooks.

## 13. Categorical Value Review

In [13]:
categorical_columns = df_clean.select_dtypes(include= ["object","str"]).columns.tolist()

categorical_summary = []

for col in categorical_columns:
    values = df_clean[col].dropna().unique()
    example_values = list(values[:10])
    
    categorical_summary.append({
        "column": col,
        "unique_count": df_clean[col].nunique(dropna=True),
        "missing_count": df_clean[col].isna().sum(),
        "missing_pct": df_clean[col].isna().mean(),
        "example_values": example_values
    })

categorical_summary = pd.DataFrame(categorical_summary).sort_values(
    ["missing_pct", "unique_count"], 
    ascending=[False, False]
)

display(categorical_summary)

,column,unique_count,missing_count,missing_pct,example_values
3,weight,9,98569,0.968585,"[[75-100), [50-75), [0-25), [100-125), [25-50)..."
9,max_glu_serum,3,96420,0.947468,"[>300, Norm, >200]"
10,A1Cresult,3,84748,0.832773,"[>7, >8, Norm]"
5,medical_specialty,72,49949,0.490822,"[Pediatrics-Endocrinology, InternalMedicine, F..."
4,payer_code,17,40256,0.395574,"[MC, MD, HM, UN, BC, SP, CP, SI, DM, CM]"
0,race,5,2273,0.022336,"[Caucasian, AfricanAmerican, Other, Asian, His..."
8,diag_3,789,1423,0.013983,"[255, V27, 403, 250, V45, 38, 486, 996, 197, 2..."
7,diag_2,748,358,0.003518,"[250.01, 250, 250.43, 157, 411, 492, 427, 198,..."
6,diag_1,716,21,0.000206,"[250.83, 276, 648, 8, 197, 414, 428, 398, 434,..."
2,age,10,0,0.000000,"[[0-10), [10-20), [20-30), [30-40), [40-50), [..."


## 14. Numeric Column Review

In [14]:
numeric_columns = df_clean.select_dtypes(include=["int64", "float64"]).columns.tolist()

numeric_summary = df_clean[numeric_columns].describe().T.reset_index()
numeric_summary = numeric_summary.rename(columns={"index": "column"})

display(numeric_summary)

,column,count,mean,std,min,25%,50%,75%,max
0,encounter_id,101766.0,1.652016e+08,1.026403e+08,12522.0,84961194.0,152388987.0,2.302709e+08,443867222.0
1,patient_nbr,101766.0,5.433040e+07,3.869636e+07,135.0,23413221.0,45505143.0,8.754595e+07,189502619.0
2,admission_type_id,101766.0,2.024006e+00,1.445403e+00,1.0,1.0,1.0,3.000000e+00,8.0
3,discharge_disposition_id,101766.0,3.715642e+00,5.280166e+00,1.0,1.0,1.0,4.000000e+00,28.0
4,admission_source_id,101766.0,5.754437e+00,4.064081e+00,1.0,1.0,7.0,7.000000e+00,25.0
5,time_in_hospital,101766.0,4.395987e+00,2.985108e+00,1.0,2.0,4.0,6.000000e+00,14.0
6,num_lab_procedures,101766.0,4.309564e+01,1.967436e+01,1.0,31.0,44.0,5.700000e+01,132.0
7,num_procedures,101766.0,1.339730e+00,1.705807e+00,0.0,0.0,1.0,2.000000e+00,6.0
8,num_medications,101766.0,1.602184e+01,8.127566e+00,1.0,10.0,15.0,2.000000e+01,81.0
9,number_outpatient,101766.0,3.693572e-01,1.267265e+00,0.0,0.0,0.0,0.000000e+00,42.0


## 15. Build Data Dictionary

In [15]:
# encounter_id and patient_nbr are retained for tracking but should not be used as predictive features.
id_columns = ["encounter_id", "patient_nbr"]
target_columns = ["readmitted", "readmitted_30d"]
sensitive_columns = ["race", "gender", "age"]

def assign_column_role(column):
    if column in id_columns:
        return "identifier"
    if column == "readmitted":
        return "original_target"
    if column == "readmitted_30d":
        return "binary_target"
    if column in sensitive_columns:
        return "sensitive_demographic"
    return "candidate_feature"

def assign_modeling_decision(column):
    if column in id_columns:
        return "exclude_from_model_keep_for_tracking"
    if column == "readmitted":
        return "exclude_from_model_original_target"
    if column == "readmitted_30d":
        return "target"
    if column in sensitive_columns:
        return "review_for_fairness_and_modeling"
    return "candidate_for_later_review"

def get_example_values(series, n=5):
    values = series.dropna().unique()
    return list(values[:n])

data_dictionary = pd.DataFrame({
    "column": df_clean.columns,
    "dtype": df_clean.dtypes.astype(str).values,
    "role": [assign_column_role(col) for col in df_clean.columns],
    "modeling_decision": [assign_modeling_decision(col) for col in df_clean.columns],
    "missing_count": df_clean.isna().sum().values,
    "missing_pct": df_clean.isna().mean().values,
    "unique_count": df_clean.nunique(dropna=False).values,
    "example_values": [get_example_values(df_clean[col]) for col in df_clean.columns]
})

# Add missingness severity to support later feature exclusion decisions.
# This does not drop columns yet; it documents which fields require special handling.

data_dictionary["missingness_flag"] = np.select(
    [
        data_dictionary["missing_pct"] >= 0.90,
        data_dictionary["missing_pct"] >= 0.40,
        data_dictionary["missing_pct"] >= 0.10,
        data_dictionary["missing_pct"] > 0
    ],
    [
        "extreme_missingness",
        "high_missingness",
        "moderate_missingness",
        "low_missingness"
    ],
    default="no_missingness"
)

# Extremely incomplete fields should not be treated as normal candidate features.
protected_roles = [
    "identifier",
    "original_target",
    "binary_target",
    "sensitive_demographic"
]

mask_extreme_missing = (
    (data_dictionary["missing_pct"] >= 0.90)
    & (~data_dictionary["role"].isin(protected_roles))
)

mask_high_missing = (
    (data_dictionary["missing_pct"] >= 0.40)
    & (data_dictionary["missing_pct"] < 0.90)
    & (~data_dictionary["role"].isin(protected_roles))
)

data_dictionary.loc[
    mask_extreme_missing,
    "modeling_decision"
] = "likely_exclude_or_use_missing_indicator_only"

data_dictionary.loc[
    mask_high_missing,
    "modeling_decision"
] = "high_missingness_review_before_modeling"


display(data_dictionary)

,column,dtype,role,modeling_decision,missing_count,missing_pct,unique_count,example_values,missingness_flag
0,encounter_id,int64,identifier,exclude_from_model_keep_for_tracking,0,0.000000,101766,"[2278392, 149190, 64410, 500364, 16680]",no_missingness
1,patient_nbr,int64,identifier,exclude_from_model_keep_for_tracking,0,0.000000,71518,"[8222157, 55629189, 86047875, 82442376, 42519267]",no_missingness
2,race,str,sensitive_demographic,review_for_fairness_and_modeling,2273,0.022336,6,"[Caucasian, AfricanAmerican, Other, Asian, His...",low_missingness
3,gender,str,sensitive_demographic,review_for_fairness_and_modeling,0,0.000000,3,"[Female, Male, Unknown/Invalid]",no_missingness
4,age,str,sensitive_demographic,review_for_fairness_and_modeling,0,0.000000,10,"[[0-10), [10-20), [20-30), [30-40), [40-50)]",no_missingness
5,weight,str,candidate_feature,likely_exclude_or_use_missing_indicator_only,98569,0.968585,10,"[[75-100), [50-75), [0-25), [100-125), [25-50)]",extreme_missingness
6,admission_type_id,int64,candidate_feature,candidate_for_later_review,0,0.000000,8,"[6, 1, 2, 3, 4]",no_missingness
7,discharge_disposition_id,int64,candidate_feature,candidate_for_later_review,0,0.000000,26,"[25, 1, 3, 6, 2]",no_missingness
8,admission_source_id,int64,candidate_feature,candidate_for_later_review,0,0.000000,17,"[1, 7, 2, 4, 5]",no_missingness
9,time_in_hospital,int64,candidate_feature,candidate_for_later_review,0,0.000000,14,"[1, 3, 2, 4, 5]",no_missingness


This notebook does not make final feature inclusion decisions. Some fields may only be available near discharge or after inpatient care events occur. Prediction timing and leakage-aware feature selection will be handled in later modeling notebooks.

## 16. Final Validation Checks

In [16]:
# These checks make sure cleaning did not accidentally change the row count or corrupt the target.
assert df_clean.shape[0] == df_raw.shape[0], "Row count changed unexpectedly."
assert "readmitted_30d" in df_clean.columns, "Missing binary target."
assert set(df_clean["readmitted_30d"].unique()) <= {0, 1}, "Target must be binary."
assert df_clean["encounter_id"].duplicated().sum() == 0, "Duplicate encounter_id found."

print("Validation checks passed.")
print("Cleaned shape:", df_clean.shape)

Validation checks passed.
Cleaned shape: (101766, 51)


## 17. Export Outputs

In [17]:
cleaned_output_path = DATA_PROCESSED / "diabetes_readmission_cleaned.csv"
# Save reference outputs used by later modeling notebooks
dictionary_output_path = MODEL_RESULTS / "data_dictionary.csv"
missing_output_path = MODEL_RESULTS / "missingness_summary.csv"

# Save cleaned starter dataset and reference files for later notebooks
df_clean.to_csv(cleaned_output_path, index=False)
data_dictionary.to_csv(dictionary_output_path, index=False)
missing_summary.to_csv(missing_output_path, index=False)

print("Saved cleaned dataset to:", cleaned_output_path.relative_to(PROJECT_ROOT))
print("Saved data dictionary to:", dictionary_output_path.relative_to(PROJECT_ROOT))
print("Saved missingness summary to:", missing_output_path.relative_to(PROJECT_ROOT))

Saved cleaned dataset to: data\processed\diabetes_readmission_cleaned.csv
Saved data dictionary to: outputs\model_results\data_dictionary.csv
Saved missingness summary to: outputs\model_results\missingness_summary.csv


## Notebook 01 Summary

This notebook completed the initial data cleaning and documentation step.

Main outputs:

- Loaded the raw UCI diabetes readmission dataset
- Confirmed encounter-level structure
- Checked duplicate encounter IDs and repeated patient IDs
- Replaced `?` missing codes with `NaN`
- Documented missingness by column
- Created the binary target `readmitted_30d`
- Flagged ID columns for exclusion from modeling
- Flagged race, gender, and age as sensitive demographic fields
- Created a data dictionary with column roles, missingness severity flags, and preliminary modeling decisions
- Exported cleaned data for EDA

No rows were dropped in this notebook. Missingness and feature exclusion decisions will be handled later during feature engineering and modeling.